## Import 

In [1]:
import math
import pickle 
import warnings
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interp
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from tableone import TableOne

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
race_path  = "Extraction/MIMICIV/Data/csvExtract/"
path_data  = "../Data/EHR/"
path_cohort= "../Data/Cohorts/"

### Read Cohorts Splits

In [3]:
with open(path_cohort + "short_icustays", "rb") as fp:   
    short_icustays = pickle.load(fp)
    
with open(path_cohort + "train_icustays", "rb") as fp:   
    train_icustays = pickle.load(fp)
    
with open(path_cohort + "valid_icustays", "rb") as fp:   
    valid_icustays = pickle.load(fp)
    
with open(path_cohort + "test_icustays", "rb") as fp:   
    test_icustays = pickle.load(fp)
    
with open(path_cohort + "icustays_ehr_text_image", "rb") as fp:   
    icustays_ehr_text_image = pickle.load(fp)
    
with open(path_cohort + "icustays_ehr_text", "rb") as fp:   
    icustays_ehr_text = pickle.load(fp)
    
with open(path_cohort + "icustays_ehr_only", "rb") as fp:   
    icustays_ehr_only = pickle.load(fp)

### Reading Data

In [5]:
df_ehr = pd.read_csv(path_data + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_ehr.head(2)

In [6]:
print(df_ehr.stay_id.nunique())
print(df_ehr.shape)

71344
(1638420, 526)


### Drop Repeated Rows & Keep Notes

In [9]:
df_note = df_ehr[['stay_id', 'radiology_note']]
df_note = df_note[df_note.radiology_note.notnull()]
df_note = df_note.drop_duplicates()

In [10]:
all_columns = list(df_ehr.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]

text_columns = ['cxr_image', 'cxr_lung', 'cxr_note', 'radiology_note', 'discharge_note', 
                'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_ehr.drop(remove_columns, axis=1, inplace=True)
df_ehr = df_ehr.drop_duplicates()

print(df_ehr.stay_id.nunique())
print(df_ehr.shape)

### Fix Age

In [14]:
df_ehr.loc[df_ehr['age'] >= 95, 'age'] = 95
df_ehr = df_ehr[df_ehr.age > 16]

### Take first hours of ICU of patients with more than 24 hour LoS

In [15]:
max_rows = df_ehr.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [16]:
def take_observation_window(df, observation_window):
    
    df = df.groupby('stay_id').head(observation_window).reset_index(drop=True)
    
    return df

In [17]:
df_ehr = take_observation_window(df_ehr, observation_window)

### Remove Short Stay

In [18]:
split_label = 'hospital_expire_flag'
label = 'icu_expire_flag'

In [19]:
df = df_ehr[~df_ehr.stay_id.isin(short_icustays)].copy()

### Variables Selection

In [20]:
selected_columns = ['subject_id', 'stay_id', 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
                    'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
                    'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                    'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
                    'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                    'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'Total GCS', 'Richmond-RAS Scale',
                    'age', 'gender', 'race', 
                    'icuLos_h', 'icu_expire_flag', 'hospital_expire_flag']

In [21]:
for col in selected_columns:
    temp_col = col + '_ind'
    
    if temp_col in list(df.columns):
        df.loc[df[temp_col] == 0, col] = np.nan

In [23]:
df = df[selected_columns]
df.head(3)

In [24]:
df_ehr = df.drop(['subject_id', 'stay_id', 'age', 'gender', 'race', 'icuLos_h', 'hospital_expire_flag'], axis=1)
df_demog = df[['subject_id', 'stay_id', 'age', 'gender', 'race', 'icuLos_h', 'icu_expire_flag']]

### Create TableOne

In [25]:
columns = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
            'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
            'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
            'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
            'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
            'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
            'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
            'Total GCS', 'Richmond-RAS Scale',
            'icu_expire_flag']

In [26]:
categorical = ['Richmond-RAS Scale']

In [27]:
IQR = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
        'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
        'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
        'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
        'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
        'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
        'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
        'Total GCS',]

In [28]:
EHR_table = TableOne(df_ehr, groupby='icu_expire_flag', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [29]:
EHR_table

Grouped by icu_expire_flag                                                                       
                                                                              Missing              Overall                  0.0                  1.0 P-Value
n                                                                                                  1442088              1351176                90912        
Heart Rate, median [Q1,Q3]                                                      91153     83.0 [72.0,96.0]     83.0 [72.0,96.0]    90.0 [76.0,105.0]  <0.001
SpO2, median [Q1,Q3]                                                           113256     97.0 [95.0,99.0]     97.0 [95.0,99.0]    97.5 [95.0,100.0]  <0.001
Oxygen Saturation, median [Q1,Q3]                                             1396599     95.0 [76.0,98.0]     95.0 [77.0,98.0]     92.0 [72.5,97.0]  <0.001
Respiratory Rate, median [Q1,Q3]                                               123903     18.5 [15.5,22.0]     18.0 [15.0,22.0]     21.0 [17.0,25.0]  <0.001
Temperature, median [Q1,Q3]                                                    994663     36.8 [36.6,37.2]     36.8 [36.6,37.2]     36.8 [36.3,37.3]  <0.001
Non Invasive Blood Pressure mean, median [Q1,Q3]                               508303     76.0 [67.0,87.0]     76.5 [67.0,87.0]     72.0 [63.0,82.0]  <0.001
Non Invasive Blood Pressure systolic, median [Q1,Q3]                           509622  116.0 [102.5,132.0]  116.0 [103.0,132.0]   109.0 [96.0,124.0]  <0.001
Non Invasive Blood Pressure diastolic, median [Q1,Q3]                          509834     63.0 [54.0,74.0]     63.0 [54.0,74.0]     60.0 [51.0,70.0]  <0.001
Glucose, median [Q1,Q3]                                                       1071154  131.0 [108.0,166.0]  131.0 [108.0,164.0]  142.0 [108.0,196.0]  <0.001
Creatinine, median [Q1,Q3]                                                    1315659        1.0 [0.7,1.6]        1.0 [0.7,1.5]        1.6 [1.0,2.6]  <0.001
Base Excess, median [Q1,Q3]                                                   1306954      -1.0 [-4.0,0.5]      -1.0 [-3.0,1.0]      -4.0 [-9.0,0.0]  <0.001
BUN, median [Q1,Q3]                                                           1316164     20.0 [13.0,35.0]     19.0 [13.0,33.0]     32.0 [20.0,52.0]  <0.001
Anion Gap, median [Q1,Q3]                                                     1317892     14.0 [12.0,17.0]     14.0 [12.0,16.0]     17.0 [14.0,21.0]  <0.001
Bicarbonate, median [Q1,Q3]                                                   1316389     23.0 [20.0,26.0]     23.0 [20.0,26.0]     20.0 [17.0,24.0]  <0.001
Lactate, median [Q1,Q3]                                                       1346375        2.0 [1.4,3.2]        1.9 [1.3,2.9]        3.3 [1.9,6.2]  <0.001
Hemoglobin, median [Q1,Q3]                                                    1274223      10.0 [8.7,11.5]      10.0 [8.7,11.5]       9.6 [8.3,11.2]  <0.001
Hematocrit, median [Q1,Q3]                                                    1274223     30.1 [26.3,34.4]     30.1 [26.3,34.5]     29.5 [25.6,34.3]  <0.001
pH, median [Q1,Q3]                                                            1283939        7.4 [7.3,7.4]        7.4 [7.3,7.4]        7.3 [7.2,7.4]  <0.001
Bilirubin, Direct, median [Q1,Q3]                                             1405290        0.4 [0.1,1.1]        0.4 [0.1,1.1]        0.6 [0.2,2.1]  <0.001
pO2, median [Q1,Q3]                                                           1306877   118.0 [80.0,195.0]   122.0 [82.0,202.0]    95.0 [66.0,142.0]  <0.001
pCO2, median [Q1,Q3]                                                          1306877     40.0 [35.5,46.0]     40.0 [36.0,46.0]     39.0 [33.0,47.0]  <0.001
AST, median [Q1,Q3]                                                           1407731    45.0 [25.0,109.0]     42.0 [24.0,99.0]    79.0 [36.0,211.0]  <0.001
ALT, median [Q1,Q3]                                                           1407130     31.0 [17.0,78.0]     30.0 [17.0,72.0]    45.

### Number of Notes

In [30]:
df_note = df_note[df_note.radiology_note.notna()]
df_note = df_note.groupby('stay_id').count().reset_index()

In [31]:
df_note.head(3)

### Static Information

In [32]:
df_demog = df_demog.groupby('stay_id').head(1)
df_demog = df_demog.merge(df_note, on='stay_id', how='left')

In [33]:
df_demog.head()

### Read Race Dictionary

In [34]:
with open(race_path + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary = pickle.load(f)

In [35]:
general_ethnicity_mapping = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNKNOWN': 'Unknown',
    'UNABLE TO OBTAIN': 'Unknown',
    'PATIENT DECLINED TO ANSWER': 'Unknown',
    
    'OTHER': 'Other',
    'MIDDLE EASTERN': 'Other',
    'MULTIPLE RACE/ETHNICITY': 'Other',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other',
    
    'ASIAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - SOUTH EAST ASIAN': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'AMERICAN INDIAN/ALASKA NATIVE': 'Native American',
    
    'BLACK/AFRICAN': 'Black/African American',
    'BLACK/CAPE VERDEAN': 'Black/African American',
    'BLACK/AFRICAN AMERICAN': 'Black/African American',
    'BLACK/CARIBBEAN ISLAND': 'Black/African American',
    
    'SOUTH AMERICAN': 'Hispanic/Latino',
    'HISPANIC OR LATINO': 'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - COLUMBIAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN': 'Hispanic/Latino'}

In [36]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['race'] = df['race'].map(inv_ethnicity_dict)
    
    return df

In [37]:
def categorize_ethnicity(df, new_mapping):
    
    df['race'] = df['race'].map(new_mapping)
    
    return df

In [38]:
df_demog = replace_ethnicity_with_names(df_demog, race_dictionary)
df_demog = categorize_ethnicity(df_demog, general_ethnicity_mapping)

In [39]:
df_demog['Ethnicity'] = 'Non Hispanic'
df_demog.loc[df_demog.race == 'Hispanic/Latino', 'Ethnicity'] = 'Hispanic'

In [40]:
df_demog.head()

In [41]:
print(df_demog.subject_id.nunique())
print(df_demog[df_demog.icu_expire_flag == 0].subject_id.nunique())
print(df_demog[df_demog.icu_expire_flag == 1].subject_id.nunique())

43733
41169
3788


In [42]:
columns = ['age', 'gender', 'race', 'Ethnicity', 'icuLos_h', 'icu_expire_flag', 'radiology_note']

categorical = ['gender', 'race', 'Ethnicity']

IQR = ['age', 'icuLos_h', 'radiology_note']

In [43]:
demog_table = TableOne(df_demog, groupby='icu_expire_flag', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [44]:
demog_table

Grouped by icu_expire_flag                                                               
                                                                         Missing           Overall               0.0                1.0 P-Value
n                                                                                            60087             56299               3788        
age, median [Q1,Q3]                                                            0  67.0 [55.0,77.0]  66.0 [55.0,77.0]   72.0 [60.0,82.0]  <0.001
gender, n (%)                  1.0                                             0      33913 (56.4)      31850 (56.6)        2063 (54.5)   0.012
                               2.0                                                    26174 (43.6)      24449 (43.4)        1725 (45.5)        
race, n (%)                    Asian                                           0        1747 (2.9)        1628 (2.9)          119 (3.1)  <0.001
                               Black/African American                                  6300 (10.5)       5948 (10.6)          352 (9.3)        
                               Hispanic/Latino                                          2277 (3.8)        2172 (3.9)          105 (2.8)        
                               Native American                                           114 (0.2)         107 (0.2)            7 (0.2)        
                               Other                                                    2096 (3.5)        1983 (3.5)          113 (3.0)        
                               Unknown                                                 6351 (10.6)       5614 (10.0)         737 (19.5)        
                               White                                                  41202 (68.6)      38847 (69.0)        2355 (62.2)        
Ethnicity, n (%)               Hispanic                                        0        2277 (3.8)        2172 (3.9)          105 (2.8)   0.001
                               Non Hispanic                                           57810 (96.2)      54127 (96.1)        3683 (97.2)        
icuLos_h, median [Q1,Q3]                                                       0  53.6 [33.2,99.5]  52.3 [32.7,95.5]  98.5 [47.3,206.4]  <0.001
radiology_note, median [Q1,Q3]                                             13301     2.0 [1.0,2.0]     2.0 [1.0,2.0]      2.0 [1.0,3.0]  <0.001

### Save Tables

In [45]:
# EHR_table.to_csv('./Results/TableONe_EHR_ICU_Mortality.csv')
# demog_table.to_csv('./Results/TableONe_DEMOG_ICU_Mortality.csv')